In [1]:
import collections
import io
import json
import pathlib
import os
import sqlite3
import sys
import urllib.parse

import cbor2
import cbor4dns.encode
import polars

from utils import list_code

DATA_COLLECTION_PATH = pathlib.Path.cwd() / "03_2_data_collection"
INPUT_DATASET_PATH = pathlib.Path.cwd() / "input_dataset"

os.environ["DATA_COLLECTION_PATH"] = str(DATA_COLLECTION_PATH)
os.environ["INPUT_DATASET_PATH"] = str(INPUT_DATASET_PATH)

# Data Collection

A graphical overview of this section.

<div>
<img title="A graphical overview of this section" src="03_2_data_collection/method.svg" width="300"/>
</div>

## Scraping JSON Responses from HTTP Archive

We used the following SQL instructions to download from the legacy BigQuery database of the HTTP archive on November 14, 2024 (no longer available):

In [2]:
list_code(DATA_COLLECTION_PATH / "big_query.sql")

CREATE TABLE cbor_paper.processed_table
PARTITION BY RANGE_BUCKET(export_id, GENERATE_ARRAY(0, 50, 1))
CLUSTER BY export_id
AS (
  SELECT
  url, req_user_agent, resp_content_length, CAST(FLOOR(50*RAND()) AS INT64) AS export_id
FROM `httparchive.summary_requests.2024_09_01_desktop`
WHERE 
  LOWER(resp_content_type) LIKE "%json%"
);
SELECT * EXCEPT(export_id)
FROM cbor_paper.processed_table
WHERE export_id IN (1, 2, 3);
SELECT * EXCEPT(export_id)
FROM cbor_paper.processed_table
WHERE export_id IN (4, 5, 6);

The last two `SELECT`s resulted in the two CSV files provided in `input_dataset/bq-results.tar.xz` (the `collect_jsons_and_dns.sh` will read from `input_dataset/bq-results.tar.xz` directly).

In [3]:
!tar --list -f "{INPUT_DATASET_PATH}/bq-results.tar.xz"

bq-results-20241115-171644-1731691029838.csv
bq-results-20241115-172202-1731691391614.csv


## Download JSONs and Query DNS

We used [curl](https://curl.se/) to download the JSONs during February 2025 from the URL in the CSVs scraped above, using the User-Agent identifier of the original request. For each URL we use the `dig` command to get the `A`, `AAAA`, and `HTTPS` records for its hostname while sniffing on the interface.

In [4]:
list_code(DATA_COLLECTION_PATH / "collect_jsons_and_dns.sh")

#! /bin/bash
#
# Copyright (C) 2025 TU Dresden
#
# Distributed under terms of the MIT license.
#

SCRIPT_PATH="$( cd -- "$( dirname -- "${BASH_SOURCE[0]}" )" &> /dev/null && pwd )"
INPUT_PATH="${INPUT_PATH:-${SCRIPT_PATH}/../input_dataset}"
OUTPUT_PATH="${OUTPUT_PATH:-${SCRIPT_PATH}/../input_dataset}"
PROCS=$(grep -c '^processor' /proc/cpuinfo)
NAMESERVER="${1:-9.9.9.9}"

"${SCRIPT_PATH}"/collect_jsons_and_dns.py -0 > "${OUTPUT_PATH}/2024-09-01-sample.csv"
tar -C "${INPUT_PATH}" -xOf "${INPUT_PATH}/"bq-results.tar.xz bq-results-20241115-171644-1731691029838.csv bq-results-20241115-172202-1731691391614.csv | \
    parallel -j"${PROCS}" --line-buffer --pipe "${SCRIPT_PATH}"/collect_jsons_and_dns.py -s "${NAMESERVER}" \
    >> "${OUTPUT_PATH}/2024-09-01-sample.csv" 2> "${OUTPUT_PATH}/2024-09-01-sample.errors.log"

In [5]:
list_code(DATA_COLLECTION_PATH / "collect_jsons_and_dns.py")

#! /usr/bin/env python3
# vim:fenc=utf-8
#
# Copyright (C) 2025 TU Dresden
#
# Distributed under terms of the MIT license.

import argparse
import csv
import io
import ipaddress
import json
import os
import re
import subprocess
import sys
import tempfile
import time
import urllib.parse

import dns.rdatatype


import pprint


CURL_PATH = os.environ.get("CURL_PATH")
DIG_PATH = os.environ.get("DIG_PATH")
TSHARK_PATH = os.environ.get("TSHARK_PATH")
DUMPCAP_PATH = os.environ.get("DUMPCAP_PATH")
FIELDS = [
    "frame.protocols",
    "dns.flags.response",
    "dns.qry.name",
    "dns.qry.type",
    "dns.resp.name",
    "dns.resp.type",
    "udp.payload",
]
FIELDS_SIMPLIFIED = {
    "frame.protocols": "protocols",
    "dns.flags.response": "is_response",
    "dns.qry.name": "query_name",
    "dns.qry.type": "query_type",
    "dns.resp.name": "response_names",
    "dns.resp.type": "response_types",
    "udp.payload": "dns_msg",
}


def canonize_name(name):
    return "." if name == "<Root>" else f"{name}."


def remove_tid(data_hex):
    return f"0000{data_hex[4:]}"


FIELDS_CAST = {
    "frame.protocols": lambda x: x,
    "dns.flags.response": lambda x: x in ["1", "True"],
    "dns.qry.name": canonize_name,
    "dns.qry.type": lambda x: dns.rdatatype.RdataType(int(x)).name,
    "dns.resp.name": lambda x: "|".join(canonize_name(n) for n in x.split("|")),
    "dns.resp.type": lambda x: "|".join(
        dns.rdatatype.RdataType(int(t)).name for t in x.split("|")
    ),
    "udp.payload": remove_tid,
}
OUTPUT_FIELDS = [
    "url",
    "http_status",
    "obj_len",
    "obj",
    "a_q_dns_msg",
    "a_q_protocols",
    "a_q_query_name",
    "a_q_query_type",
    "a_q_response_names",
    "a_q_response_types",
    "a_r_dns_msg",
    "a_r_protocols",
    "a_r_query_name",
    "a_r_query_type",
    "a_r_response_names",
    "a_r_response_types",
    "aaaa_q_dns_msg",
    "aaaa_q_protocols",
    "aaaa_q_query_name",
    "aaaa_q_query_type",
    "aaaa_q_response_names",
    "aaaa_q_response_types",
    "aaaa_r_dns_msg",
    "aaaa_r_protocols",
    "aaaa_r_query_name",
    "aaaa_r_query_type",
    "aaaa_r_response_names",
    "aaaa_r_response_types",
    "https_q_dns_msg",
    "https_q_protocols",
    "https_q_query_name",
    "https_q_query_type",
    "https_q_response_names",
    "https_q_response_types",
    "https_r_dns_msg",
    "https_r_protocols",
    "https_r_query_name",
    "https_r_query_type",
    "https_r_response_names",
    "https_r_response_types",
]


class UnableToGetError(Exception):
    pass


class UnableToResolveError(Exception):
    pass


def curl(url, user_agent):
    try:
        response = subprocess.check_output(
            [
                CURL_PATH,
                "-w",
                "\n\t\n\t\n\tcanary\t%{response_code}",
                "-o",
                "-",
                "--connect-timeout",
                "10",
                "-s",
                "-A",
                user_agent,
                "-L",
                url
            ]
        )
    except subprocess.CalledProcessError as exc:
        raise UnableToGetError(f"Could not fetch URL: {exc}") from exc
    try:
        response = response.split(b"\n\t\n\t\n\tcanary\t")
        http_status, obj = int(response[1]), json.loads(response[0])
    except json.JSONDecodeError as exc:
        raise UnableToGetError(f"Unable to decode '{response}': {exc}") from exc
    return http_status, obj


def dig(url, sniff_interface=None, nameserver=None):
    if nameserver is None:
        nameserver = ipaddress.ip_address("9.9.9.9")
    extra_args = []
    if sniff_interface is not None:
        extra_args = ["-i", sniff_interface]
    url_parse = urllib.parse.urlparse(url)
    with tempfile.NamedTemporaryFile() as fp:
        proc = subprocess.Popen(
            [
                DUMPCAP_PATH,
                "-w",
                fp.name
            ] + extra_args,
        )
        time.sleep(1)
        responded_records = set()
        for record in [

To start a detached TMUX session running this script **in background of the Docker setup**, run the following command (with the UV setup you might need to step into the virtualenv first, adding, e.g., `. '${DATA_COLLECTION_PATH}'/../.env/bin/activate;` before the `INPUT_PATH='${INPUT_DATASET_PATH}' ${DATA_COLLECTION_PATH}/collect_jsons_and_dns.sh`). You can change the name server used by changing the `'9.9.9.9'` argument.

In [6]:
%%bash

tmux new-session -s "collect_jsons" -d "INPUT_PATH='${INPUT_DATASET_PATH}' '${DATA_COLLECTION_PATH}'/collect_jsons_and_dns.sh '9.9.9.9'"

To attach that TMUX session, run the following commant in a Terminal in your Jupyter Lab.

```sh
tmux attach -t "collect_jsons"
```

There will be no output though (`stdout` goes to `input_datasets/2024-09-01-sample.csv` and `stderr` to `input_datasets/2024-09-01-sample.errors.log`).

To kill the TMUX session you can use the following command into a Terminal in your Jupyter Lab.

```sh
tmux send-keys -t "collect_jsons" C-c
```

**When you collected enough objects _and killed_ the session** you can compress the resulting CSV:

In [7]:
%%bash

PROCS=$(grep -c '^processor' /proc/cpuinfo)

pigz -fp "${PROCS}" "${INPUT_DATASET_PATH}/2024-09-01-sample.csv"

## Convert JSON and DNS to CBOR and Store to Database

We load the previously generated CSV and convert the JSONs in there to CBOR using `cbor2` and convert DNS to `application/dns+cbor` using `cbor4dns`. We use polars to parallelize this conversion efficiently on a per row basis.

In [8]:
def convert_to_cbor(obj_json):
    obj = json.loads(obj_json, object_pairs_hook=collections.OrderedDict)
    return cbor2.dumps(obj, canonical=True)

def query_to_json(url):
    url_parts = urllib.parse.urlsplit(url)
    query = urllib.parse.parse_qs(url_parts.query)
    if query:
        for key in query:
            for i, value in enumerate(query[key]):
                try:
                    query[key][i] = int(value)
                except ValueError:
                    try:
                        query[key][i] = float(value)
                    except ValueError:
                        pass
            if len(query[key]) == 1:
                query[key] = query[key][0]
        return json.dumps(query, separators=(',', ':'), sort_keys=True)
    return None

def remove_query(url):
    url_parts = urllib.parse.urlsplit(url)
    assert not url_parts.fragment
    return urllib.parse.urlunsplit(
        (url_parts.scheme, url_parts.netloc, url_parts.path, None, None)
    )

def encode_cbor_dns(msgs):
    if isinstance(msgs, bytes):
        msg = msgs
        orig_query = None
    else:
        msg = msgs["classic_response"]
        orig_query = msgs["cbor_query"]        
    with io.BytesIO() as file:
        encoder = cbor4dns.encode.Encoder(
            file,
            packed=False,
            always_omit_question=False,
        )
        encoder.encode(msg, orig_query)
        return file.getvalue()

df = polars.read_csv(
    INPUT_DATASET_PATH / "2024-09-01-sample.csv.gz",
    separator=";",
    quote_char="'"
).with_row_index(name="id", offset=1)
objects = df[["id", "url", "http_status", "obj"]].rename({"obj": "json"})
objects[["cbor"]] = objects.select(polars.col("json").map_elements(convert_to_cbor, return_dtype=polars.Binary))
objects[["json_query"]] = objects.select(polars.col("url").map_elements(query_to_json, return_dtype=polars.String))
objects[["url_wo_query"]] = objects.select(polars.col("url").map_elements(remove_query, return_dtype=polars.String))
objects[["cbor_query"]] = objects.select(polars.col("json_query").map_elements(convert_to_cbor, return_dtype=polars.Binary))

dns = polars.concat(
    [
        df.filter(~polars.col("https_q_dns_msg").is_null())[
            [
                "https_q_query_name",
                "https_q_query_type",
                "https_q_response_names",
                "https_q_response_types",
                "https_q_dns_msg",
                "https_r_response_names",
                "https_r_response_types",
                "https_r_dns_msg",
                "url",
                "id",
            ]
        ].rename(
            {
                "https_q_query_name": "name",
                "https_q_query_type": "type",
                "https_q_dns_msg": "classic_query",
                "https_q_response_names": "query_add_names",
                "https_q_response_types": "query_add_types",
                "https_r_dns_msg": "classic_response",
                "https_r_response_names": "response_names",
                "https_r_response_types": "response_types",
            }
        ),
        df.filter(~polars.col("aaaa_q_dns_msg").is_null())[
            [
                "aaaa_q_query_name",
                "aaaa_q_query_type",
                "aaaa_q_response_names",
                "aaaa_q_response_types",
                "aaaa_q_dns_msg",
                "aaaa_r_response_names",
                "aaaa_r_response_types",
                "aaaa_r_dns_msg",
                "url",
                "id",
            ]
        ].rename(
            {
                "aaaa_q_query_name": "name",
                "aaaa_q_query_type": "type",
                "aaaa_q_dns_msg": "classic_query",
                "aaaa_q_response_names": "query_add_names",
                "aaaa_q_response_types": "query_add_types",
                "aaaa_r_dns_msg": "classic_response",
                "aaaa_r_response_names": "response_names",
                "aaaa_r_response_types": "response_types",
            }
        ),
        df.filter(~polars.col("a_q_dns_msg").is_null())[
            [
                "a_q_query_name",
                "a_q_query_type",
                "a_q_response_names",
                "a_q_response_types",
                "a_q_dns_msg",
                "a_r_response_names",
                "a_r_response_types",
                "a_r_dns_msg",
                "url",
                "id",
            ]
        ].rename(
            {
                "a_q_query_name": "name",
                "a_q_query_type": "type",
                "a_q_dns_msg": "classic_query",
                "a_q_response_names": "query_add_names",
                "a_q_response_types": "query_add_types",
                "a_r_dns_msg": "classic_response",
                "a_r_response_names": "response_names",
                "a_r_response_types": "response_types",
            }
        ),
    ]
).sort(["name", "type", "classic_query", "classic_response"]).rename({"id": "obj_id"})
dns[["classic_query"]] = dns.select(polars.col("classic_query").map_elements(bytes.fromhex, return_dtype=polars.Binary))
dns[["classic_response"]] = dns.select(polars.col("classic_response").map_elements(bytes.fromhex, return_dtype=polars.Binary))
dns[["cbor_query"]] = dns.select(polars.col("classic_query").map_elements(encode_cbor_dns, return_dtype=polars.Binary))
dns = dns.with_columns(polars.struct("classic_response", "cbor_query").map_elements(encode_cbor_dns, return_dtype=polars.Binary).alias('cbor_response'))
dns = dns[
    [
        "name",
        "type",
        "query_add_names",
        "query_add_types",
        "classic_query",
        "cbor_query",
        "response_names",
        "response_types",
        "classic_response",
        "cbor_response",
        "url",
        "obj_id",
    ]
]

Now store to SQLite database to not require later conversion of binaries and more efficiently map relations of objects and DNS.

In [9]:
DB_FILE = INPUT_DATASET_PATH / "2024-09-01-sample.sqlite3"

with sqlite3.connect(DB_FILE) as conn:
    cur = conn.cursor()
    cur.execute("""CREATE TABLE objects (
        id INTEGER PRIMARY KEY,
        url TEXT,
        http_status INTEGER,
        json TEXT,
        cbor BLOB,
        json_query TEXT,
        url_wo_query TEXT,
        cbor_query BLOB
    );""")
    cur.execute("""CREATE TABLE dns (
        id INTEGER PRIMARY KEY,
        name TEXT,
        type TEXT,
        query_add_names TEXT,
        query_add_types TEXT,
        classic_query BLOB,
        cbor_query BLOB,
        response_names TEXT,
        response_types TEXT,
        classic_response BLOB,
        cbor_response BLOB,
        url TEXT,
        obj_id INTEGER,
        FOREIGN KEY (obj_id) REFERENCES objects(id)
    );""")
    conn.commit()


objects.write_database("objects", f"sqlite:///{DB_FILE}", if_table_exists="append")
dns.write_database("dns", f"sqlite:///{DB_FILE}", if_table_exists="append")

with sqlite3.connect(DB_FILE) as conn:
    cur = conn.cursor()
    cur.execute("CREATE INDEX objects_url ON objects (url);")
    cur.execute("CREATE INDEX objects_url_json_query ON objects (url_wo_query, json_query);")
    cur.execute("CREATE INDEX objects_url_cbor_query ON objects (url_wo_query, cbor_query);")
    cur.execute("CREATE INDEX dns_url ON dns (url);")
    cur.execute("CREATE INDEX dns_name_type ON dns (name, type);")
    conn.commit()